## 手写LSTM
### 输入门
$ i_t = \sigma(W_{xi} x_t + W_{hi} h_{t-1} + b_i) $
### 遗忘门
$ f_t = \sigma(W_{xf} x_t + W_{hf} h_{t-1} + b_f) $
### 输出门
$ o_t = \sigma(W_{xo} x_t + W_{ho} h_{t-1} + b_o) $
### 候选记忆单元
$ c_t = tanh(W_{xc} x_t + W_{hc} h_{t-1} + b_c) $<br>
$c_t = f_t \cdot c_{t-1} + i_t \cdot c_t$<br>
$h_t = o_t \cdot tanh(c_t) $


In [1]:
import torch
from torch import nn


class LSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.W_xi = nn.Parameter(torch.randn(input_size, hidden_size))
        self.W_hi = nn.Parameter(torch.randn(hidden_size, hidden_size))
        self.b_i = nn.Parameter(torch.zeros(hidden_size))

        self.W_xf = nn.Parameter(torch.randn(input_size, hidden_size))
        self.W_hf = nn.Parameter(torch.randn(hidden_size, hidden_size))
        self.b_f = nn.Parameter(torch.zeros(hidden_size))

        self.W_xo = nn.Parameter(torch.randn(input_size, hidden_size))
        self.W_ho = nn.Parameter(torch.randn(hidden_size, hidden_size))
        self.b_o = nn.Parameter(torch.zeros(hidden_size))

        self.W_xc = nn.Parameter(torch.randn(input_size, hidden_size))
        self.W_hc = nn.Parameter(torch.randn(hidden_size, hidden_size))
        self.b_c = nn.Parameter(torch.zeros(hidden_size))

    def forward(self, x, states):
        h_prev, c_prev = states
        # 输入门
        i_t = torch.sigmoid(torch.mm(x, self.W_xi) + torch.mm(h_prev, self.W_hi) + self.b_i)
        # 遗忘门
        f_t = torch.sigmoid(torch.mm(x, self.W_xf) + torch.mm(h_prev, self.W_hf) + self.b_f)
        # 输出门
        o_t = torch.sigmoid(torch.mm(x, self.W_xo) + torch.mm(h_prev, self.W_ho) + self.b_o)
        # 候选状态
        c_t_tilde = torch.tanh(torch.mm(x, self.W_xc) + torch.mm(h_prev, self.W_hc) + self.b_c)
        c_t = f_t * c_prev + i_t * c_t_tilde
        h_t = o_t * torch.tanh(c_t)
        return h_t, c_t


class LSTM(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.cell = LSTMCell(input_size, hidden_size)

    def forward(self, x, hidden=None, cell=None):
        seq_len, batch_size, input_size = x.shape
        if hidden is None:
            hidden = torch.zeros(batch_size, self.hidden_size)
            cell = torch.zeros(batch_size, self.hidden_size)
        outputs = []
        for t in range(seq_len):
            hidden, cell = self.cell(x[t], (hidden, cell))
            outputs.append(hidden)
        return outputs, (hidden, cell)


input_size = 32
hidden_size = 5
batch_size = 1
seq_len = 2

model = LSTM(input_size, hidden_size)
x = torch.randn(seq_len, batch_size, input_size)
outputs, (hidden, cell) = model(x)
print(outputs)
print(hidden)
print(cell)

[tensor([[-0.0226, -0.5257,  0.7252,  0.4737,  0.1062]], grad_fn=<MulBackward0>), tensor([[-1.0502e-01, -7.2520e-01, -4.1156e-02,  5.6130e-02, -2.2135e-04]],
       grad_fn=<MulBackward0>)]
tensor([[-1.0502e-01, -7.2520e-01, -4.1156e-02,  5.6130e-02, -2.2135e-04]],
       grad_fn=<MulBackward0>)
tensor([[-1.1452e-01, -9.2540e-01, -8.4658e-01,  2.7698e-01, -4.0116e-04]],
       grad_fn=<AddBackward0>)


### LSTM文本生成实战

In [2]:
text = """
臣密言：臣以险衅，夙遭闵凶。生孩六月，慈父见背；行年四岁，舅夺母志。
祖母刘愍臣孤弱，躬亲抚养。
臣少多疾病，九岁不行，零丁孤苦，至于成立。
既无伯叔，终鲜兄弟，门衰祚薄，晚有儿息。
外无期功强近之亲，内无应门五尺之僮，茕茕孑立，形影相吊。
而刘夙婴疾病，常在床蓐，臣侍汤药，未曾废离。
"""

words = set(text)
vocab_size = len(words)
word_to_index = {word: i for i, word in enumerate(words)}
index_to_word = {i: word for i, word in enumerate(words)}

print(word_to_index)

{'苦': 0, '既': 1, '不': 2, '蓐': 3, '形': 4, '弟': 5, '亲': 6, '臣': 7, '衅': 8, '生': 9, '六': 10, '愍': 11, '弱': 12, '：': 13, '相': 14, '五': 15, '晚': 16, '孩': 17, '近': 18, '茕': 19, '，': 20, '舅': 21, '废': 22, '四': 23, '之': 24, '于': 25, '在': 26, '尺': 27, '慈': 28, '抚': 29, '父': 30, '密': 31, '兄': 32, '外': 33, '曾': 34, '刘': 35, '母': 36, '九': 37, '月': 38, '养': 39, '以': 40, '影': 41, '汤': 42, '立': 43, '行': 44, '躬': 45, '儿': 46, '少': 47, '祚': 48, '床': 49, '离': 50, '成': 51, '伯': 52, '零': 53, '期': 54, '有': 55, '终': 56, '门': 57, '内': 58, '孤': 59, '遭': 60, '；': 61, '婴': 62, '鲜': 63, '无': 64, '衰': 65, '多': 66, '夺': 67, '至': 68, '应': 69, '未': 70, '强': 71, '病': 72, '志': 73, '。': 74, '息': 75, '险': 76, '吊': 77, '侍': 78, '年': 79, '\n': 80, '言': 81, '闵': 82, '背': 83, '岁': 84, '常': 85, '僮': 86, '功': 87, '而': 88, '药': 89, '见': 90, '丁': 91, '孑': 92, '薄': 93, '祖': 94, '疾': 95, '叔': 96, '夙': 97, '凶': 98}


In [3]:
from torch.utils.data import Dataset
import torch

SEQ_LEN = 5
BATCH_SIZE = 1
HIDDEN_SIZE = 128
EMBEDDING_SIZE = 128


class TextDataset(Dataset):
    def __init__(self, text, seq_len):
        self.text = text
        self.seq_len = seq_len
        self.data = [word_to_index[ch] for ch in text]

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, index):
        input_seq = self.data[index:index + self.seq_len]
        target_seq = self.data[index + 1:index + self.seq_len + 1]
        return torch.tensor(input_seq), torch.tensor(target_seq)


dataset = TextDataset(text, SEQ_LEN)
train_loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
print(dataset.data)

[80, 7, 31, 81, 13, 7, 40, 76, 8, 20, 97, 60, 82, 98, 74, 9, 17, 10, 38, 20, 28, 30, 90, 83, 61, 44, 79, 23, 84, 20, 21, 67, 36, 73, 74, 80, 94, 36, 35, 11, 7, 59, 12, 20, 45, 6, 29, 39, 74, 80, 7, 47, 66, 95, 72, 20, 37, 84, 2, 44, 20, 53, 91, 59, 0, 20, 68, 25, 51, 43, 74, 80, 1, 64, 52, 96, 20, 56, 63, 32, 5, 20, 57, 65, 48, 93, 20, 16, 55, 46, 75, 74, 80, 33, 64, 54, 87, 71, 18, 24, 6, 20, 58, 64, 69, 57, 15, 27, 24, 86, 20, 19, 19, 92, 43, 20, 4, 41, 14, 77, 74, 80, 88, 35, 97, 62, 95, 72, 20, 85, 26, 49, 3, 20, 7, 78, 42, 89, 20, 70, 34, 22, 50, 74, 80]


In [4]:
import torch
import torch.nn as nn


class LSTM(nn.Module):
    def __init__(self, vocab_size, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, batch_first=True, num_layers=2)
        self.out_linear = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, state=None):
        embedding = self.embedding(x)
        outputs, (hidden, cell) = self.lstm(embedding, state)
        outputs = self.out_linear(outputs)
        return outputs, (hidden, cell)

In [6]:
model = LSTM(vocab_size, EMBEDDING_SIZE, HIDDEN_SIZE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(100):
    for i, (input_seq, target_seq) in enumerate(train_loader):
        output, _ = model(input_seq)
        loss = criterion(
            output.view(-1, vocab_size),
            target_seq.view(-1)
        )
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if i % 10 == 0:
            print(f'Epoch [{epoch + 1}/100], Step [{i + 10}/{len(train_loader)}], Loss: {loss.item():.8f}')

Epoch [1/100], Step [10/140], Loss: 4.60231304
Epoch [1/100], Step [20/140], Loss: 4.57557535
Epoch [1/100], Step [30/140], Loss: 4.51135015
Epoch [1/100], Step [40/140], Loss: 4.48546696
Epoch [1/100], Step [50/140], Loss: 4.35818052
Epoch [1/100], Step [60/140], Loss: 4.21846581
Epoch [1/100], Step [70/140], Loss: 4.11821461
Epoch [1/100], Step [80/140], Loss: 4.45694113
Epoch [1/100], Step [90/140], Loss: 4.40230083
Epoch [1/100], Step [100/140], Loss: 4.21651697
Epoch [1/100], Step [110/140], Loss: 3.47236633
Epoch [1/100], Step [120/140], Loss: 3.70234108
Epoch [1/100], Step [130/140], Loss: 4.16507673
Epoch [1/100], Step [140/140], Loss: 3.78434873
Epoch [2/100], Step [10/140], Loss: 3.78049707
Epoch [2/100], Step [20/140], Loss: 2.95241642
Epoch [2/100], Step [30/140], Loss: 3.70932627
Epoch [2/100], Step [40/140], Loss: 3.64012456
Epoch [2/100], Step [50/140], Loss: 3.75203705
Epoch [2/100], Step [60/140], Loss: 2.77538300
Epoch [2/100], Step [70/140], Loss: 2.61867762
Epoch [2

In [9]:

model.eval()


def generate_text(context, step, temperature=0.8):
    words = [word for word in context]
    state = None

    for _ in range(step):
        input_seq = torch.tensor([word_to_index[word] for word in words[-1:]])
        input_seq = torch.LongTensor(input_seq)
        input_seq = input_seq.view(1, -1)

        with torch.no_grad():
            output, state = model(input_seq, state)
            last_output = output[0, -1, :]
            probs = torch.softmax(last_output / temperature, dim=-1)
            result_index = torch.multinomial(probs, 1).item()
            result = index_to_word[result_index]
            words.append(result)
    return ''.join(words)


print(generate_text('臣密言：', 5, 0.1))

臣密言：臣以险衅，
